## Project Overview

This project is to make an ai agent that can prevent intrusion attack in networks. We will sequentially work with NSL-KDD --> CICIDS2017 dataset to complete this progress. 

## Note

- NSL-KDD is a data set suggested to solve some of the inherent problems of the KDD'99 data set which are mentioned in [1]. Although, this new version of the KDD data set still suffers from some of the problems discussed by McHugh [2] and may not be a perfect representative of existing real networks, because of the lack of public data sets for network-based IDSs, we believe it still can be applied as an effective benchmark data set to help researchers compare different intrusion detection methods. Furthermore, the number of records in the NSL-KDD train and test sets are reasonable. This advantage makes it affordable to run the experiments on the complete set without the need to randomly select a small portion. Consequently, evaluation results of different research work will be consistent and comparable.

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [46]:
current_folder = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

train_data_path = project_root/"dataset"/"KDDTrain+.txt"
test_data_path = project_root/"dataset"/"KDDTest+.txt"

In [47]:
column_names = [
    "duration", "protocol_type", "service", "flag",
    "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
    "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label", "difficulty_level"
]

In [48]:
train_df = pd.read_csv(train_data_path, names=column_names, header=None)
test_df = pd.read_csv(test_data_path, names=column_names, header=None)

In [49]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0,0,0,0.05,0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0,0.60,0.88,0,0,0,0,0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0,0,1,1,0,0,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1,0,0.03,0.04,0.03,0.01,0,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1,0,0,0,0,0,0,0,normal,21


In [50]:
train_df.duration.unique()

array([    0,  5607,   507, ...,  5430, 11680,   679], shape=(2981,))

In [51]:
train_df.shape

(125973, 43)

In [52]:
test_df.shape

(22544, 43)

In [53]:
train_df.isna().sum().sum()

np.int64(0)

In [54]:
test_df.isna().sum().sum()

np.int64(0)

In [55]:
train_df.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty_level
count,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973,...,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973
mean,287.14,45566.74,19779.11,0.00,0.02,0.00,0.20,0.00,0.40,0.28,...,115.65,0.52,0.08,0.15,0.03,0.28,0.28,0.12,0.12,19.50
std,2604.52,5870331.18,4021269.15,0.01,0.25,0.01,2.15,0.05,0.49,23.94,...,110.70,0.45,0.19,0.31,0.11,0.44,0.45,0.31,0.32,2.29
min,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
25%,0,0,0,0,0,0,0,0,0,0,...,10,0.05,0,0,0,0,0,0,0,18
50%,0,44,0,0,0,0,0,0,0,0,...,63,0.51,0.02,0,0,0,0,0,0,20
75%,0,276,516,0,0,0,0,0,1,0,...,255,1,0.07,0.06,0.02,1,1,0,0,21
max,42908,1379963888,1309937401,1,3,3,77,5,1,7479,...,255,1,1,1,1,1,1,1,1,21


In [56]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  str    
 2   service                      125973 non-null  str    
 3   flag                         125973 non-null  str    
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13  root_shell

In [57]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 43 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     22544 non-null  int64  
 1   protocol_type                22544 non-null  str    
 2   service                      22544 non-null  str    
 3   flag                         22544 non-null  str    
 4   src_bytes                    22544 non-null  int64  
 5   dst_bytes                    22544 non-null  int64  
 6   land                         22544 non-null  int64  
 7   wrong_fragment               22544 non-null  int64  
 8   urgent                       22544 non-null  int64  
 9   hot                          22544 non-null  int64  
 10  num_failed_logins            22544 non-null  int64  
 11  logged_in                    22544 non-null  int64  
 12  num_compromised              22544 non-null  int64  
 13  root_shell                 

In [58]:
train_df.label.unique()

<StringArray>
[         'normal',         'neptune',     'warezclient',         'ipsweep',
       'portsweep',        'teardrop',            'nmap',           'satan',
           'smurf',             'pod',            'back',    'guess_passwd',
       'ftp_write',        'multihop',         'rootkit', 'buffer_overflow',
            'imap',     'warezmaster',             'phf',            'land',
      'loadmodule',             'spy',            'perl']
Length: 23, dtype: str

In [59]:
train_df.difficulty_level.value_counts()

difficulty_level
21    62557
18    20667
20    19339
19    10284
15     3990
17     3074
16     2393
12      729
14      674
11      641
13      451
10      253
9       194
7       118
8       106
6        96
5        81
4        79
0        66
3        65
1        62
2        54
Name: count, dtype: int64

## Difficulty Level Analysis
- No model can't predict 66 records.

In [60]:
train_df.shape

(125973, 43)

In [61]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0,0,0,0.05,0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0,0.60,0.88,0,0,0,0,0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0,0,1,1,0,0,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1,0,0.03,0.04,0.03,0.01,0,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1,0,0,0,0,0,0,0,normal,21


In [62]:
train_df[train_df['duration'] == 0]

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0,0,0,0.05,0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0,0.60,0.88,0,0,0,0,0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0,0,1,1,0,0,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1,0,0.03,0.04,0.03,0.01,0,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1,0,0,0,0,0,0,0,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125967,0,tcp,http,SF,359,375,0,0,0,0,...,1,0,0.33,0.04,0.33,0,0,0,normal,18
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0,0,1,1,0,0,neptune,20
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0,0,0.72,0,0.01,0,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0,0,1,1,0,0,neptune,20


In [63]:
train_df['protocol_type'].unique()

<StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str

In [64]:
train_df['flag'].unique()

<StringArray>
['SF', 'S0', 'REJ', 'RSTR', 'SH', 'RSTO', 'S1', 'RSTOS0', 'S3', 'S2', 'OTH']
Length: 11, dtype: str

In [65]:
train_df['protocol_type'].dtype
train_df['duration'].dtype
train_df['dst_host_same_srv_rate'].dtype

dtype('float64')

In [66]:
# features = [col for col in train_df.columns if train_df[col].dtype == 'int64' or train_df[col].dtype == 'float64']
features = train_df.select_dtypes(include=['number']).columns.tolist()
features

['duration',
 'src_bytes',
 'dst_bytes',
 'land',
 'wrong_fragment',
 'urgent',
 'hot',
 'num_failed_logins',
 'logged_in',
 'num_compromised',
 'root_shell',
 'su_attempted',
 'num_root',
 'num_file_creations',
 'num_shells',
 'num_access_files',
 'num_outbound_cmds',
 'is_host_login',
 'is_guest_login',
 'count',
 'srv_count',
 'serror_rate',
 'srv_serror_rate',
 'rerror_rate',
 'srv_rerror_rate',
 'same_srv_rate',
 'diff_srv_rate',
 'srv_diff_host_rate',
 'dst_host_count',
 'dst_host_srv_count',
 'dst_host_same_srv_rate',
 'dst_host_diff_srv_rate',
 'dst_host_same_src_port_rate',
 'dst_host_srv_diff_host_rate',
 'dst_host_serror_rate',
 'dst_host_srv_serror_rate',
 'dst_host_rerror_rate',
 'dst_host_srv_rerror_rate',
 'difficulty_level']

In [67]:
binary_features = [feature for feature in features if set(train_df[feature].dropna().unique())=={0,1}]
print(train_df[binary_features].value_counts().sum())
binary_features

125973


['land', 'logged_in', 'root_shell', 'is_host_login', 'is_guest_login']

In [68]:
count_features = [feature for feature in features if "count" in feature]
print(train_df[count_features].value_counts().sum())
count_features

125973


['count', 'srv_count', 'dst_host_count', 'dst_host_srv_count']

In [69]:
rate_features = [feature for feature in features if "rate" in feature]
train_df[rate_features]

,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,0,0,0,0,1,0,0,0.17,0.03,0.17,0,0,0,0.05,0
1,0,0,0,0,0.08,0.15,0,0,0.60,0.88,0,0,0,0,0
2,1,1,0,0,0.05,0.07,0,0.10,0.05,0,0,1,1,0,0
3,0.20,0.20,0,0,1,0,0,1,0,0.03,0.04,0.03,0.01,0,0.01
4,0,0,0,0,1,0,0.09,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,1,1,0,0,0.14,0.06,0,0.10,0.06,0,0,1,1,0,0
125969,0,0,0,0,1,0,0,0.96,0.01,0.01,0,0,0,0,0
125970,0,0,0,0,1,0,0,0.12,0.06,0,0,0.72,0,0.01,0
125971,1,1,0,0,0.06,0.05,0,0.03,0.05,0,0,1,1,0,0


In [70]:
train_df[['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','num_failed_logins','logged_in','count','serror_rate','label','difficulty_level']]

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,num_failed_logins,logged_in,count,serror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,2,0,normal,20
1,0,udp,other,SF,146,0,0,0,0,13,0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,123,1,neptune,19
3,0,tcp,http,SF,232,8153,0,0,1,5,0.20,normal,21
4,0,tcp,http,SF,199,420,0,0,1,30,0,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,184,1,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,2,0,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,1,1,0,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,144,1,neptune,20


In [71]:
train_df.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty_level
count,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973,...,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973
mean,287.14,45566.74,19779.11,0.00,0.02,0.00,0.20,0.00,0.40,0.28,...,115.65,0.52,0.08,0.15,0.03,0.28,0.28,0.12,0.12,19.50
std,2604.52,5870331.18,4021269.15,0.01,0.25,0.01,2.15,0.05,0.49,23.94,...,110.70,0.45,0.19,0.31,0.11,0.44,0.45,0.31,0.32,2.29
min,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
25%,0,0,0,0,0,0,0,0,0,0,...,10,0.05,0,0,0,0,0,0,0,18
50%,0,44,0,0,0,0,0,0,0,0,...,63,0.51,0.02,0,0,0,0,0,0,20
75%,0,276,516,0,0,0,0,0,1,0,...,255,1,0.07,0.06,0.02,1,1,0,0,21
max,42908,1379963888,1309937401,1,3,3,77,5,1,7479,...,255,1,1,1,1,1,1,1,1,21


In [72]:
#iqr function
def stats(data)->float:
    data = sorted(data)
    n = len(data)

    def get_median(sub_list):
        length = len(sub_list)
        mid = length//2
        if length%2==0:
            return (sub_list[mid-1]+sub_list[mid])/2
        else:
            return sub_list[mid]
    
    mid_index = n//2

    if n%2==0:
        lower = data[:mid_index]
        upper = data[mid_index:]
    else:
        lower = data[:mid_index]
        upper = data[mid_index+1:] #for odd dataset

    q1 = get_median(lower)
    q3 = get_median(upper)

    iqr = q3 - q1

    return q1, q3, iqr



In [73]:
q1, q3, iqr = stats(train_df['duration'].sort_values())
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")

lower_fench = q1 - (1.5*iqr)
upper_fench = q3 + (1.5*iqr)

Q1: 0.0, Q3: 0.0, IQR: 0.0


## Inter-quartile Range Analysis
- While Q1, Q3 and IQR is 0, then the lower and upper fence is also 0.

In [74]:
max_duration_zero = (train_df[train_df['duration']==0].value_counts().sum()*100)/(train_df['duration'].value_counts().sum())
max_duration_zero

np.float64(92.04750224254404)

In [75]:
max_duration = (train_df[train_df['duration']>0].value_counts().sum()*100)/(train_df['duration'].value_counts().sum())
max_duration

np.float64(7.952497757455963)

## Zero Duration Analysis

In [76]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0,0,0,0.05,0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0,0.60,0.88,0,0,0,0,0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0,0,1,1,0,0,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1,0,0.03,0.04,0.03,0.01,0,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1,0,0,0,0,0,0,0,normal,21


In [77]:
skew_map = {}

for key, value in train_df[features].items():
    skew_map[key] = value.skew()

kurt_map = {}

for key, value in train_df[features].items():
    kurt_map[key] = value.kurt()

In [78]:
stats_df = pd.DataFrame({
    "skewness":pd.Series(skew_map),
    "kurtosis":pd.Series(kurt_map)
})
stats_df.index.name = "features"

In [79]:
stats_df

,skewness,kurtosis
features,,
duration,11.88,156.08
src_bytes,190.67,39354.12
dst_bytes,290.05,90941.73
land,70.97,5034.12
wrong_fragment,11.46,130.83
urgent,149.91,24967.32
hot,12.59,168.01
num_failed_logins,53.76,3869.07
logged_in,0.43,-1.82


## Custom function to calculate skewness and kurtosis

In [80]:
train_df.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty_level
count,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973,...,125973,125973,125973,125973,125973,125973,125973,125973,125973,125973
mean,287.14,45566.74,19779.11,0.00,0.02,0.00,0.20,0.00,0.40,0.28,...,115.65,0.52,0.08,0.15,0.03,0.28,0.28,0.12,0.12,19.50
std,2604.52,5870331.18,4021269.15,0.01,0.25,0.01,2.15,0.05,0.49,23.94,...,110.70,0.45,0.19,0.31,0.11,0.44,0.45,0.31,0.32,2.29
min,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
25%,0,0,0,0,0,0,0,0,0,0,...,10,0.05,0,0,0,0,0,0,0,18
50%,0,44,0,0,0,0,0,0,0,0,...,63,0.51,0.02,0,0,0,0,0,0,20
75%,0,276,516,0,0,0,0,0,1,0,...,255,1,0.07,0.06,0.02,1,1,0,0,21
max,42908,1379963888,1309937401,1,3,3,77,5,1,7479,...,255,1,1,1,1,1,1,1,1,21


In [81]:
duration_mean = sum(train_df['duration'].values)/sum(train_df['duration'].value_counts())
duration_mean

np.float64(287.1446500440571)

In [82]:
def skewness(df: pd.DataFrame)->pd.DataFrame:
    import pandas as pd
    features = df.select_dtypes(include=['number']).columns.tolist()

    skew_results = {}
    for col in features:
        col_data = df[col].values
        n = len(col_data)

        mean = sum(col_data)/n

        varience_sum= sum((i-mean)**2 for i in col_data)
        varience = varience_sum/(n-1)
        std = varience**0.5

        if std==0:
            skew = 0
        else:
            skew = (n/((n-1)*(n-2)))*(sum((i-mean)**3 for i in col_data)/std**3)
            skew_results[col] = skew
    
    skew_df = pd.DataFrame({
        "skewness":pd.Series(skew_results)
    })

    return skew_df




In [83]:
skew_data = skewness(train_df)

In [84]:
skew_data

,skewness
duration,11.88
src_bytes,190.67
dst_bytes,290.05
land,70.97
wrong_fragment,11.46
urgent,149.91
hot,12.59
num_failed_logins,53.76
logged_in,0.43
num_compromised,250.11
